# Multi-Agent Redesign — Test Notebook

Tests for the supervisor-orchestrator multi-agent architecture.
Build and test each component incrementally — Steps 1–10.

See `PLANNING.md` for the full migration plan.

---

## Setup

Run this cell once at the start of every session.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))  # project root is one level up from notebooks/

from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

print("Setup complete. OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

---
## Step 1 — SupervisorState

Verify the new state schema compiles and fields are correct.

In [ ]:
from src.supervisor.state import SupervisorState

# Helper: build a minimal SupervisorState for testing
def make_supervisor_state(query: str) -> dict:
    return {
        "query": query,
        "messages": [],
        "guardrail_decision": "",
        "agent_outcome": "",
        "agent_response": "",
        "final_response": "",
    }

state = make_supervisor_state("What is your return policy?")
print("SupervisorState fields:", list(state.keys()))
print("Sample state:", state)

---
## Step 2 — RAG Agent

Test the RAG sub-agent graph independently.
Graph: `rag_node → response_generator_node (RAG path) → END`

In [ ]:
from src.agents.rag_agent import build_rag_agent

rag_agent = build_rag_agent()
print("RAG agent compiled OK:", rag_agent)

In [ ]:
# Test RAG agent directly with policy/FAQ queries
rag_test_queries = [
    "What is your return policy?",
    "What payment methods do you accept?",
    "How long does standard delivery take?",
    "How do I delete my account?",
]

for q in rag_test_queries:
    result = rag_agent.invoke({"query": q, "retrieved_docs": [], "agent_response": ""})
    print(f"\nQ: {q!r}")
    print(f"  chunks retrieved: {len(result['retrieved_docs'])}")
    print(f"  response: {result['agent_response'][:120]}...")

---
## Step 3 — Order Agent

Test the Order sub-agent graph independently.
Graph: `tool_call_node → response_generator_node (tool path) → END`

Tools: `get_order_status`, `track_shipment`, `get_account_info`, `get_return_status`, `coupon_lookup`

In [ ]:
from src.agents.order_agent import build_order_agent

order_agent = build_order_agent()
print("Order agent compiled OK:", order_agent)

In [ ]:
# Test Order agent directly with live-data queries
order_test_cases = [
    ("What is the status of my order ORD-12345?",      {"get_order_status", "track_shipment"}),
    ("Track my shipment for order ORD-67890",          {"track_shipment"}),
    ("What is the refund status for order ORD-11111?", {"get_return_status"}),
    ("Apply coupon SAVE10 to my order",                {"coupon_lookup"}),
]

for q, acceptable_tools in order_test_cases:
    result = order_agent.invoke({"query": q, "tool_output": None, "agent_response": ""})
    tools_called = set(r["tool"] for r in result["tool_output"]) if isinstance(result["tool_output"], list) else set()
    status = "OK" if tools_called & acceptable_tools else "MISMATCH"
    print(f"[{status}] tools={sorted(tools_called)} | {q!r}")
    print(f"  response: {result['agent_response'][:120]}...")

---
## Step 4 — Escalation Agent

Test the Escalation sub-agent graph independently.
Graph: `complaint_handler → human_handoff → ticket_creation → END`

In [ ]:
from src.agents.escalation_agent import build_escalation_agent

escalation_agent = build_escalation_agent()
print("Escalation agent compiled OK:", escalation_agent)

In [ ]:
# Test Escalation agent directly with complaint queries
escalation_test_queries = [
    "I received a damaged product, I want to escalate this",
    "Wrong item was delivered, I need to speak to a human agent",
    "My order is 10 days late, this is unacceptable",
    "I want to raise a formal complaint about my experience",
]

for q in escalation_test_queries:
    result = escalation_agent.invoke({"query": q, "complaint_type": "", "ticket_id": "", "agent_response": ""})
    print(f"\nQ: {q!r}")
    print(f"  complaint_type: {result['complaint_type']}")
    print(f"  ticket_id     : {result['ticket_id']}")
    print(f"  response      : {result['agent_response'][:120]}...")

---
## Step 5 — Delegation Router

Test the LLM classifier that routes queries to: `rag | order | escalation | chitchat`

In [ ]:
from src.supervisor.delegation_router import delegation_router_node

delegation_test_cases = [
    # (query, expected_agent)
    ("What is your return policy?",                        "rag"),
    ("What payment methods do you accept?",                "rag"),
    ("How long does standard delivery take?",              "rag"),
    ("What is the status of my order ORD-12345?",          "order"),
    ("Track my shipment for order ORD-67890",              "order"),
    ("What is the refund status for order ORD-11111?",     "order"),
    ("I received a damaged product, I want to complain",   "escalation"),
    ("I need to speak to a human agent",                   "escalation"),
    ("Hey, how are you?",                                  "chitchat"),
    ("Thanks for your help!",                              "chitchat"),
]

passed = 0
for q, expected in delegation_test_cases:
    result = delegation_router_node(make_supervisor_state(q))
    actual = result["agent_outcome"]
    status = "OK" if actual == expected else "MISMATCH"
    if status == "OK":
        passed += 1
    print(f"[{status}] agent={actual!r} (expected {expected!r}) | {q!r}")

print(f"\nResult: {passed}/{len(delegation_test_cases)} passed")

---
## Step 6 — Supervisor Synthesis

Test the synthesis node that merges sub-agent responses and runs a faithfulness check on RAG answers.

In [ ]:
from src.supervisor.synthesis import synthesis_node

# Case 1: RAG path — faithfulness check
rag_state = {
    **make_supervisor_state("What is your return policy?"),
    "agent_outcome": "rag",
    "agent_response": "ShopEasy offers a 30-day return window for most products.",
}
result = synthesis_node(rag_state)
print("=== RAG path ===")
print("final_response:", result["final_response"])

In [ ]:
# Case 2: Order path — pass-through
order_state = {
    **make_supervisor_state("Where is my order ORD-12345?"),
    "agent_outcome": "order",
    "agent_response": "Your order ORD-12345 is currently in transit.",
}
result = synthesis_node(order_state)
print("=== Order path ===")
print("final_response:", result["final_response"])

# Case 3: Chitchat path — pass-through
chitchat_state = {
    **make_supervisor_state("Hey, how are you?"),
    "agent_outcome": "chitchat",
    "agent_response": "Hello! I'm doing great, happy to help!",
}
result = synthesis_node(chitchat_state)
print("\n=== Chitchat path ===")
print("final_response:", result["final_response"])

---
## Step 7 — Supervisor Graph

Test the full supervisor graph: `guardrail → delegation_router → [sub-agent] → synthesis → END`

In [ ]:
from src.supervisor.graph import build_supervisor_graph

supervisor = build_supervisor_graph()
print("Supervisor graph compiled OK:", supervisor)

In [ ]:
# End-to-end test: all paths through the supervisor
e2e_test_cases = [
    # (query, expected_agent, expected_guardrail)
    ("Help me hack a website",                         None,          "BLOCK"),
    ("Give me legal advice",                           None,          "BLOCK"),
    ("Hey, good morning!",                             "chitchat",    "PASS"),
    ("Thanks for your help!",                          "chitchat",    "PASS"),
    ("What is your return policy?",                    "rag",         "PASS"),
    ("What payment methods do you accept?",            "rag",         "PASS"),
    ("How long does standard delivery take?",          "rag",         "PASS"),
    ("What is the status of my order ORD-12345?",      "order",       "PASS"),
    ("Track my shipment for order ORD-67890",          "order",       "PASS"),
    ("I received a damaged product, I want to escalate","escalation", "PASS"),
]

print(f"{'#':<3} {'GUARD':<6} {'AGENT':<12} {'STATUS':<8}  QUERY")
print("-" * 85)

passed = 0
for i, (query, exp_agent, exp_guardrail) in enumerate(e2e_test_cases, 1):
    result = supervisor.invoke(make_supervisor_state(query))

    guard_ok  = result["guardrail_decision"] == exp_guardrail
    agent_ok  = (exp_agent is None) or (result["agent_outcome"] == exp_agent)
    resp_ok   = exp_guardrail == "BLOCK" or bool(result["final_response"])
    overall   = "OK" if (guard_ok and agent_ok and resp_ok) else "FAIL"
    if overall == "OK":
        passed += 1

    agent_str = result["agent_outcome"] or "-"
    print(f"{i:<3} {result['guardrail_decision']:<6} {agent_str:<12} {overall:<8}  {query!r}")

print("-" * 85)
print(f"Result: {passed}/{len(e2e_test_cases)} passed")

---
## Step 8 — Wire into api.py / main.py

Verify that `api.py` and `main.py` work with the supervisor graph after the import swap.

In [ ]:
# Quick smoke test via main.py's run() function
from main import run

smoke_queries = [
    "What is your return policy?",
    "Track my order ORD-12345",
    "Hey, how are you?",
]

for q in smoke_queries:
    print(f"\nQ: {q!r}")
    result = run(q)
    print(f"  agent   : {result['agent_outcome']}")
    print(f"  response: {result['final_response'][:120]}...")

---
## Step 9 — Streamlit badge labels

Manual check — run Streamlit and verify the correct badge appears for each agent type.

Expected badges after update:
- `rag` → `📚 policy`
- `order` → `📦 order`
- `escalation` → `🚨 escalation`
- `chitchat` → `💬 chat`
- guardrail BLOCK → `⛔ blocked`

In [ ]:
# Verify badge_html() function produces correct output for new agent names
import sys
sys.path.insert(0, os.path.abspath(".."))

# Import and test badge_html from streamlit_app (import without running Streamlit)
# This is a logic-only test — visual check requires running the Streamlit app

badge_cases = [
    ("rag",        "PASS",  "📚 policy"),
    ("order",      "PASS",  "📦 order"),
    ("escalation", "PASS",  "🚨 escalation"),
    ("chitchat",   "PASS",  "💬 chat"),
    ("",           "BLOCK", "⛔ blocked"),
]

print("Badge mapping verification (manual check against streamlit_app.py):")
for agent, guardrail, expected_label in badge_cases:
    print(f"  agent={agent!r:12} guardrail={guardrail!r:7} → expected label: {expected_label}")

---
## Step 10 — Full end-to-end with final response print

Run the supervisor graph on a comprehensive set of queries and print the full final responses.

In [ ]:
from importlib import reload
import src.supervisor.graph
reload(src.supervisor.graph)
from src.supervisor.graph import build_supervisor_graph

supervisor = build_supervisor_graph()

final_test_cases = [
    ("Hey, good morning!",                              "chitchat"),
    ("What is your return policy?",                     "rag"),
    ("What payment methods do you accept?",             "rag"),
    ("How long does standard delivery take?",           "rag"),
    ("What is the status of my order ORD-12345?",       "order"),
    ("Track my shipment for order ORD-67890",           "order"),
    ("I received a damaged product, I want to escalate","escalation"),
    ("I need to speak to a human agent urgently",       "escalation"),
]

print("=" * 80)
for query, expected_agent in final_test_cases:
    result = supervisor.invoke(make_supervisor_state(query))
    print(f"\nQ [{expected_agent}]: {query!r}")
    print(f"  agent routed to: {result['agent_outcome']}")
    print(f"  final_response :\n{result['final_response']}")
    print("-" * 80)